# 04 — Nominal Models

Three-class nominal benchmark for `No / Less / Most`.

**Representations:** RDKit 2D descriptors and Morgan/ECFP4 (2048 bits).  
**Outer evaluation:** fixed parent-grouped random folds and scaffold-grouped folds from Notebook 03.  
**Models:** Logistic Regression, RBF SVM, Random Forest, Histogram Gradient Boosting.

Runtime is kept practical with 3-fold grouped inner CV, compact grids, 300-tree RF, and SVM without probability calibration.


## 1. Imports and paths

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, recall_score,
    confusion_matrix, mean_absolute_error, cohen_kappa_score,
)

import warnings

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*Liblinear failed to converge.*",
)

warnings.filterwarnings(
    "ignore",
    message=".*Solver terminated early.*",
)


RANDOM_STATE = 42
INNER_SPLITS = 3

candidate_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (p for p in candidate_roots if (p / "data/features/feature_ids.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Notebook 03 outputs not found.")

FEATURE_DIR = PROJECT_ROOT / "data/features"
SPLIT_DIR = PROJECT_ROOT / "data/splits"
TABLE_DIR = PROJECT_ROOT / "results/tables"
PRED_DIR = PROJECT_ROOT / "results/predictions"
FIG_DIR = PROJECT_ROOT / "results/figures/nominal"

for p in [TABLE_DIR, PRED_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


Project root: /home/gman/Documents/BS-Bioinformatics/dilirank2-prediction


## 2. Load fixed features and modelling population

In [2]:
feature_ids = pd.read_csv(FEATURE_DIR / "feature_ids.csv")
descriptor_file = pd.read_csv(FEATURE_DIR / "rdkit_descriptors.csv")
population = pd.read_csv(SPLIT_DIR / "concern_modelling_population.csv")
morgan_npz = np.load(FEATURE_DIR / "morgan_ecfp4_2048.npz", allow_pickle=True)
X_morgan_all = morgan_npz["X"]

assert np.array_equal(feature_ids["feature_row"].to_numpy(), np.arange(len(feature_ids)))
assert len(X_morgan_all) == len(feature_ids)

descriptor_columns = [c for c in descriptor_file.columns if c not in {"feature_row", "LTKBID"}]

X_descriptor_df = (
    descriptor_file.sort_values("feature_row")[descriptor_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
)

FLOAT32_LIMIT = np.finfo(np.float32).max
too_large = X_descriptor_df.abs() > FLOAT32_LIMIT

print("Descriptor values outside float32 range:", int(too_large.to_numpy().sum()))
print("Affected descriptor columns:", int(too_large.any(axis=0).sum()))

# Keep the descriptor column, but mark only impossible individual values as missing.
X_descriptor_df = X_descriptor_df.mask(too_large, np.nan)

model_rows = population["feature_row"].to_numpy(dtype=int)
X_descriptor = X_descriptor_df.iloc[model_rows].to_numpy(dtype=np.float64)
X_morgan = X_morgan_all[model_rows].astype(np.uint8)

y_class = population["DILI_class"].astype(str).to_numpy()
y_order = population["DILI_order"].astype(int).to_numpy()

CLASS_ORDER = ["No", "Less", "Most"]
CLASS_TO_ORDER = {"No": 0, "Less": 1, "Most": 2}

display(population["DILI_class"].value_counts().reindex(CLASS_ORDER).to_frame("count"))
print("RDKit:", X_descriptor.shape, "| Morgan:", X_morgan.shape)


Descriptor values outside float32 range: 20
Affected descriptor columns: 1


,count
DILI_class,
No,350
Less,330
Most,206


RDKit: (886, 217) | Morgan: (886, 2048)


## 3. Leakage-safe preprocessing

RDKit: training-fold median imputation → training-fold constant removal → scaling for LR/SVM.  
Morgan: training-fold constant-bit removal only.


In [3]:
def make_pipeline(representation, model_name, classifier):
    steps = []

    if representation == "rdkit":
        steps += [
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("variance", VarianceThreshold(0.0)),
        ]
        if model_name in {"logistic_regression", "svm"}:
            steps.append(("scaler", StandardScaler()))

    elif representation == "morgan_ecfp4":
        steps.append(("variance", VarianceThreshold(0.0)))

    else:
        raise ValueError(representation)

    steps.append(("clf", classifier))
    return Pipeline(steps)


## 4. Models and compact tuning grids

In [4]:
MODEL_SPECS = {
    "logistic_regression": {
        "estimator": LogisticRegression(
            class_weight="balanced", max_iter=5000, solver="lbfgs",
            random_state=RANDOM_STATE
        ),
        "param_grid": {"clf__C": [0.1, 1.0, 10.0]},
    },
    "svm": {
        "estimator": SVC(
            kernel="rbf", class_weight="balanced",
            random_state=RANDOM_STATE
        ),
        "param_grid": {
            "clf__C": [1.0, 4.0],
            "clf__gamma": ["scale", "auto"],
        },
    },
    "random_forest": {
        "estimator": RandomForestClassifier(
            n_estimators=300, class_weight="balanced_subsample",
            n_jobs=-1, random_state=RANDOM_STATE
        ),
        "param_grid": {
            "clf__max_depth": [None, 12],
            "clf__min_samples_leaf": [1, 3],
            "clf__max_features": ["sqrt"],
        },
    },
    "hist_gradient_boosting": {
        "estimator": HistGradientBoostingClassifier(
            class_weight="balanced", random_state=RANDOM_STATE
        ),
        "param_grid": {
            "clf__learning_rate": [0.05, 0.10],
            "clf__max_leaf_nodes": [15, 31],
            "clf__l2_regularization": [0.0],
        },
    },
}


## 5. Metrics and grouped inner CV

In [5]:
def compute_metrics(y_true, y_pred):
    yt = np.array([CLASS_TO_ORDER[x] for x in y_true])
    yp = np.array([CLASS_TO_ORDER[x] for x in y_pred])

    recalls = recall_score(
        y_true, y_pred, labels=CLASS_ORDER,
        average=None, zero_division=0
    )

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "recall_No": recalls[0],
        "recall_Less": recalls[1],
        "recall_Most": recalls[2],
        "ordinal_mae": mean_absolute_error(yt, yp),
        "quadratic_weighted_kappa": cohen_kappa_score(yt, yp, weights="quadratic"),
    }

def make_inner_cv():
    return StratifiedGroupKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )


## 6. Quick sanity check

In [6]:
outer = population["random_fold"].astype(int).to_numpy()
test_mask = outer == 0
train_mask = ~test_mask

rows = []
for model_name, spec in MODEL_SPECS.items():
    pipe = make_pipeline("rdkit", model_name, clone(spec["estimator"]))
    pipe.fit(X_descriptor[train_mask], y_class[train_mask])
    pred = pipe.predict(X_descriptor[test_mask])
    ba = balanced_accuracy_score(y_class[test_mask], pred)
    rows.append({"model": model_name, "balanced_accuracy": ba, "status": "OK"})
    print(model_name, "OK", f"BA={ba:.3f}")

display(pd.DataFrame(rows))


logistic_regression OK BA=0.486
svm OK BA=0.467
random_forest OK BA=0.466
hist_gradient_boosting OK BA=0.418


,model,balanced_accuracy,status
0,logistic_regression,0.485570,OK
1,svm,0.467100,OK
2,random_forest,0.465945,OK
3,hist_gradient_boosting,0.417605,OK


## 7. Nested nominal benchmark

In [7]:
REPRESENTATIONS = {
    "rdkit": X_descriptor,
    "morgan_ecfp4": X_morgan,
}
SPLIT_REGIMES = {
    "random": {"fold_column": "random_fold", "group_column": "parent_group"},
    "scaffold": {"fold_column": "scaffold_fold", "group_column": "scaffold_group"},
}

fold_result_rows = []
prediction_rows = []

checkpoint_folds = TABLE_DIR / "nominal_fold_results_checkpoint.csv"
checkpoint_preds = PRED_DIR / "nominal_predictions_checkpoint.csv"

for representation_name, X in REPRESENTATIONS.items():
    for split_name, info in SPLIT_REGIMES.items():

        outer_folds = population[info["fold_column"]].astype(int).to_numpy()
        groups_all = population[info["group_column"]].astype(str).to_numpy()

        for model_name, spec in MODEL_SPECS.items():
            print("\n" + "=" * 80)
            print(representation_name, split_name, model_name)

            for outer_fold in sorted(np.unique(outer_folds)):
                print("Outer fold", outer_fold)

                test_mask = outer_folds == outer_fold
                train_mask = ~test_mask

                pipeline = make_pipeline(
                    representation_name,
                    model_name,
                    clone(spec["estimator"]),
                )

                search = GridSearchCV(
                    pipeline,
                    spec["param_grid"],
                    scoring="balanced_accuracy",
                    cv=make_inner_cv(),
                    n_jobs=-1,
                    refit=True,
                    error_score="raise",
                    verbose=1,
                )

                start = time.perf_counter()
                search.fit(
                    X[train_mask],
                    y_class[train_mask],
                    groups=groups_all[train_mask],
                )
                elapsed = time.perf_counter() - start

                y_test = y_class[test_mask]
                y_pred = search.predict(X[test_mask])
                metrics = compute_metrics(y_test, y_pred)

                fold_result_rows.append({
                    "representation": representation_name,
                    "split_regime": split_name,
                    "model": model_name,
                    "outer_fold": int(outer_fold),
                    "n_train": int(train_mask.sum()),
                    "n_test": int(test_mask.sum()),
                    "inner_best_balanced_accuracy": float(search.best_score_),
                    "best_params": json.dumps(search.best_params_, sort_keys=True),
                    "fit_seconds": elapsed,
                    **metrics,
                })

                tp = population.loc[test_mask].reset_index(drop=True)
                for i in range(len(tp)):
                    prediction_rows.append({
                        "feature_row": int(tp.loc[i, "feature_row"]),
                        "LTKBID": tp.loc[i, "LTKBID"],
                        "CompoundName": tp.loc[i, "CompoundName"],
                        "true_class": y_test[i],
                        "true_order": CLASS_TO_ORDER[y_test[i]],
                        "predicted_class": y_pred[i],
                        "predicted_order": CLASS_TO_ORDER[y_pred[i]],
                        "representation": representation_name,
                        "split_regime": split_name,
                        "model": model_name,
                        "outer_fold": int(outer_fold),
                    })

                pd.DataFrame(fold_result_rows).to_csv(checkpoint_folds, index=False)
                pd.DataFrame(prediction_rows).to_csv(checkpoint_preds, index=False)

                print(
                    f"BA={metrics['balanced_accuracy']:.3f} | "
                    f"Macro-F1={metrics['macro_f1']:.3f} | "
                    f"Inner BA={search.best_score_:.3f} | "
                    f"{elapsed:.1f}s"
                )



rdkit random logistic_regression
Outer fold 0
Fitting 3 folds for each of 3 candidates, totalling 9 fits
BA=0.490 | Macro-F1=0.485 | Inner BA=0.470 | 2.4s
Outer fold 1
Fitting 3 folds for each of 3 candidates, totalling 9 fits
BA=0.488 | Macro-F1=0.485 | Inner BA=0.482 | 1.9s
Outer fold 2
Fitting 3 folds for each of 3 candidates, totalling 9 fits
BA=0.470 | Macro-F1=0.467 | Inner BA=0.491 | 0.4s
Outer fold 3
Fitting 3 folds for each of 3 candidates, totalling 9 fits
BA=0.496 | Macro-F1=0.483 | Inner BA=0.465 | 0.4s
Outer fold 4
Fitting 3 folds for each of 3 candidates, totalling 9 fits
BA=0.500 | Macro-F1=0.487 | Inner BA=0.457 | 0.4s

rdkit random svm
Outer fold 0
Fitting 3 folds for each of 4 candidates, totalling 12 fits
BA=0.467 | Macro-F1=0.463 | Inner BA=0.503 | 0.3s
Outer fold 1
Fitting 3 folds for each of 4 candidates, totalling 12 fits
BA=0.549 | Macro-F1=0.543 | Inner BA=0.481 | 0.3s
Outer fold 2
Fitting 3 folds for each of 4 candidates, totalling 12 fits
BA=0.468 | Macro-F1

## 8. Save and summarize

In [8]:
fold_results = pd.DataFrame(fold_result_rows)
predictions = pd.DataFrame(prediction_rows)

fold_results.to_csv(TABLE_DIR / "nominal_fold_results.csv", index=False)
predictions.to_csv(PRED_DIR / "nominal_predictions.csv", index=False)

summary = (
    fold_results
    .groupby(["representation", "split_regime", "model"], as_index=False)
    .agg(
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        accuracy_mean=("accuracy", "mean"),
        ordinal_mae_mean=("ordinal_mae", "mean"),
        qwk_mean=("quadratic_weighted_kappa", "mean"),
        recall_No_mean=("recall_No", "mean"),
        recall_Less_mean=("recall_Less", "mean"),
        recall_Most_mean=("recall_Most", "mean"),
        mean_fit_seconds=("fit_seconds", "mean"),
    )
    .sort_values(["split_regime", "balanced_accuracy_mean"], ascending=[True, False])
)

summary.to_csv(TABLE_DIR / "nominal_summary.csv", index=False)
display(summary.round(3))


,representation,split_regime,model,balanced_accuracy_mean,balanced_accuracy_std,macro_f1_mean,macro_f1_std,accuracy_mean,ordinal_mae_mean,qwk_mean,recall_No_mean,recall_Less_mean,recall_Most_mean,mean_fit_seconds
2,morgan_ecfp4,random,random_forest,0.537,0.037,0.529,0.036,0.537,0.554,0.421,0.623,0.439,0.549,5.801
3,morgan_ecfp4,random,svm,0.524,0.049,0.523,0.045,0.536,0.546,0.401,0.597,0.524,0.452,2.208
1,morgan_ecfp4,random,logistic_regression,0.517,0.044,0.513,0.038,0.521,0.570,0.387,0.591,0.464,0.495,5.589
11,rdkit,random,svm,0.505,0.038,0.498,0.036,0.505,0.621,0.321,0.560,0.439,0.515,0.336
10,rdkit,random,random_forest,0.501,0.050,0.502,0.049,0.520,0.571,0.349,0.597,0.530,0.374,3.210
8,rdkit,random,hist_gradient_boosting,0.498,0.050,0.498,0.051,0.516,0.580,0.345,0.600,0.509,0.384,34.828
0,morgan_ecfp4,random,hist_gradient_boosting,0.497,0.051,0.496,0.050,0.509,0.594,0.335,0.583,0.482,0.428,87.224
9,rdkit,random,logistic_regression,0.489,0.011,0.481,0.008,0.488,0.654,0.266,0.506,0.461,0.500,1.113
6,morgan_ecfp4,scaffold,random_forest,0.512,0.051,0.505,0.047,0.520,0.609,0.320,0.617,0.451,0.466,4.302
5,morgan_ecfp4,scaffold,logistic_regression,0.502,0.013,0.500,0.012,0.516,0.595,0.332,0.626,0.457,0.422,6.897


## 9. Random-vs-scaffold gap

In [9]:
gap = summary.pivot_table(
    index=["representation", "model"],
    columns="split_regime",
    values="balanced_accuracy_mean",
).reset_index()

if {"random", "scaffold"}.issubset(gap.columns):
    gap["random_minus_scaffold"] = gap["random"] - gap["scaffold"]

display(gap.round(3))


split_regime,representation,model,random,scaffold,random_minus_scaffold
0,morgan_ecfp4,hist_gradient_boosting,0.497,0.497,0.000
1,morgan_ecfp4,logistic_regression,0.517,0.502,0.015
2,morgan_ecfp4,random_forest,0.537,0.512,0.026
3,morgan_ecfp4,svm,0.524,0.493,0.032
4,rdkit,hist_gradient_boosting,0.498,0.500,-0.002
5,rdkit,logistic_regression,0.489,0.490,-0.001
6,rdkit,random_forest,0.501,0.502,-0.001
7,rdkit,svm,0.505,0.458,0.047


## Notebook Complete

This is the matched nominal benchmark for comparison with Notebook 05.
